# KNN Algorithm - Churn Dataset

In [17]:
import pandas as pd # Veri analizi ve manipülasyonu için kullanılan bir kütüphane
import seaborn as sns # Veri görselleştirme için kullanılan bir kütüphane
import matplotlib.pyplot as plt # Grafik çizimi için kullanılan bir kütüphane

from sklearn.preprocessing import LabelBinarizer, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler # Veriyi sayısal hale getirmek ve ölçeklendirmek için kullanılan araçlar
from sklearn.compose import ColumnTransformer # Verilerin farklı türlerine farklı ön işleme adımları uygulamak için kullanılan bir araç
from sklearn.model_selection import train_test_split # Veriyi eğitim ve test setlerine bölmek için kullanılan bir fonksiyon
from sklearn.neighbors import KNeighborsClassifier # KNN algoritmasını uygulamak için kullanılan bir sınıflandırıcı
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, f1_score # Model performansını değerlendirmek için kullanılan metrikler

In [18]:
df = pd.read_pickle(
    filepath_or_buffer='data/churndata.pkl'
) # Veri setini 'data/churndata.pkl' dosyasından yükler ve df adlı bir DataFrame oluşturur. Bu DataFrame, müşteri kaybı (churn) analizi için kullanılacak verileri içerir.

df.head()

,id,months,offer,phone,multiple,internet_type,gb_mon,security,backup,protection,...,unlimited,contract,paperless,payment,monthly,total_revenue,satisfaction,churn_value,churn_score,cltv
0,8779-QRDMV,1,None,No,No,DSL,8,No,No,Yes,...,No,Month-to-Month,Yes,Bank Withdrawal,39.65,59.65,3,1,91,5433
1,7495-OOKFY,8,Offer E,Yes,Yes,Fiber Optic,17,No,Yes,No,...,Yes,Month-to-Month,Yes,Credit Card,80.65,1024.10,3,1,69,5302
2,1658-BYGOY,18,Offer D,Yes,Yes,Fiber Optic,52,No,No,No,...,Yes,Month-to-Month,Yes,Bank Withdrawal,95.45,1910.88,2,1,81,3179
3,4598-XLKNJ,25,Offer C,Yes,No,Fiber Optic,12,No,Yes,Yes,...,Yes,Month-to-Month,Yes,Bank Withdrawal,98.50,2995.07,2,1,88,5337
4,4846-WHAFZ,37,Offer C,Yes,Yes,Fiber Optic,14,No,No,No,...,Yes,Month-to-Month,Yes,Bank Withdrawal,76.50,3102.36,2,1,67,2793


In [19]:
df.shape # Veri setinin satır ve sütun sayısını gösterir. df.shape[0] satır sayısını, df.shape[1] ise sütun sayısını verir.

(7043, 21)

In [20]:
df.info() # Veri setindeki sütunların veri tiplerini ve eksik değerlerin sayısını gösterir.

<class 'pandas.core.frame.DataFrame'>
Int64Index: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             7043 non-null   object 
 1   months         7043 non-null   int64  
 2   offer          7043 non-null   object 
 3   phone          7043 non-null   object 
 4   multiple       7043 non-null   object 
 5   internet_type  7043 non-null   object 
 6   gb_mon         7043 non-null   int64  
 7   security       7043 non-null   object 
 8   backup         7043 non-null   object 
 9   protection     7043 non-null   object 
 10  support        7043 non-null   object 
 11  unlimited      7043 non-null   object 
 12  contract       7043 non-null   object 
 13  paperless      7043 non-null   object 
 14  payment        7043 non-null   object 
 15  monthly        7043 non-null   float64
 16  total_revenue  7043 non-null   float64
 17  satisfaction   7043 non-null   int64  
 18  churn_va

In [21]:
df.isnull().sum() # Veri setindeki her bir sütunda eksik değerlerin sayısını gösterir.

id               0
months           0
offer            0
phone            0
multiple         0
internet_type    0
gb_mon           0
security         0
backup           0
protection       0
support          0
unlimited        0
contract         0
paperless        0
payment          0
monthly          0
total_revenue    0
satisfaction     0
churn_value      0
churn_score      0
cltv             0
dtype: int64

In [22]:
df.drop(
    columns=['id', 'phone', 'total_revenue', 'cltv', 'churn_score'],
    axis=1,
    inplace=True
)

df.shape

(7043, 16)

In [23]:
round(df.describe(), 2) # Veri setindeki sayısal sütunların temel istatistiklerini gösterir. round() fonksiyonu, istatistiklerin daha okunabilir olması için sonuçları 2 ondalık basamağa yuvarlar.

,months,gb_mon,monthly,satisfaction,churn_value
count,7043.00,7043.00,7043.00,7043.00,7043.00
mean,32.39,20.52,64.76,3.24,0.27
std,24.54,20.42,30.09,1.20,0.44
min,1.00,0.00,18.25,1.00,0.00
25%,9.00,3.00,35.50,3.00,0.00
50%,29.00,17.00,70.35,3.00,0.00
75%,55.00,27.00,89.85,4.00,1.00
max,72.00,85.00,118.75,5.00,1.00


In [24]:
df.describe(include='object') # Veri setindeki kategorik sütunların temel istatistiklerini gösterir. include='object' ifadesi, sadece kategorik sütunları seçmek için kullanılır. Bu, her bir kategorik sütunun benzersiz değerlerini, en sık görülen değeri (top), ve bu değerin frekansını (freq) içerir.

,offer,multiple,internet_type,security,backup,protection,support,unlimited,contract,paperless,payment
count,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043
unique,6,2,4,2,2,2,2,2,3,2,3
top,None,No,Fiber Optic,No,No,No,No,Yes,Month-to-Month,Yes,Bank Withdrawal
freq,3877,4072,3035,5024,4614,4621,4999,4745,3610,4171,3909


In [25]:
# Benzersiz değerlerin sayısını göstermek, her bir kategorik sütunun kaç farklı kategori içerdiğini anlamamıza yardımcı olur. Bu, özellikle modelleme aşamasında hangi sütunların one-hot encoding veya label encoding gerektirebileceğini belirlemek için önemlidir.
# Ayrıca, benzersiz değerlerin sayısı çok yüksek olan kategorik sütunlar, modelin karmaşıklığını artırabilir ve overfitting riskini yükseltebilir. Bu nedenle, benzersiz değerlerin sayısını bilmek, veri ön işleme stratejilerini planlamak için kritik bir adımdır.
# Sonuç olarak, benzersiz değerlerin sayısını göstermek, veri setindeki kategorik sütunların yapısını ve modelleme sürecinde nasıl ele alınması gerektiğini anlamamıza yardımcı olur.

df_uniques = pd.DataFrame(
    data=[[col, len(df[col].unique())]for col in df.columns], # Her bir sütun için benzersiz değerlerin sayısını hesaplar ve bir liste oluşturur. Bu liste, her sütun adı ve o sütundaki benzersiz değerlerin sayısını içeren alt listelerden oluşur.
    columns=['Variable Name', 'Unique Values Count'] # Oluşturulan listeyi bir DataFrame'e dönüştürür ve sütun adlarını 'Variable Name' ve 'Unique Values Count' olarak belirler.
).set_index('Variable Name') # 'Variable Name' sütununu DataFrame'in indeksine dönüştürür, böylece her bir satırın indeks olarak sütun adını kullanır ve 'Unique Values Count' sütunu benzersiz değerlerin sayısını içerir.

df_uniques # Her bir sütunun benzersiz değer sayısını gösterir.

,Unique Values Count
Variable Name,
months,72
offer,6
multiple,2
internet_type,4
gb_mon,50
security,2
backup,2
protection,2
support,2


In [26]:
# İkili kategorik değişkenler, modelleme sürecinde genellikle daha basit işlemler gerektirir, bu nedenle bu tür değişkenlerin sayısını bilmek, veri ön işleme stratejilerini planlamak için önemlidir.
# Ayrıca, ikili kategorik değişkenler, modelin performansını etkileyebilir, bu nedenle bu tür değişkenlerin sayısını bilmek, modelleme sürecinde nasıl ele alınması gerektiğini anlamamıza yardımcı olur.

binary_variables = list(df_uniques[df_uniques['Unique Values Count'] == 2].index) # Benzersiz değer sayısı 2 olan sütunları seçer ve bu sütunların adlarını bir liste olarak saklar. 
# Bu, ikili (binary) kategorik değişkenleri tanımlamak için kullanılır, çünkü bu tür değişkenler sadece iki farklı kategori içerir (örneğin, 'Evet'/'Hayır', '0'/'1' gibi). 
# Bu liste, daha sonra bu sütunlara özel işlemler uygulamak için kullanılabilir, örneğin one-hot encoding veya label encoding gibi.

binary_variables # İkili (binary) kategorik değişkenlerin listesini gösterir.

['multiple',
 'security',
 'backup',
 'protection',
 'support',
 'unlimited',
 'paperless',
 'churn_value']

In [27]:
# Çok kategorili değişkenler, modelin karmaşıklığını artırabilir, bu nedenle bu tür değişkenlerin sayısını bilmek, veri ön işleme stratejilerini planlamak için önemlidir.
# Ayrıca, çok kategorili değişkenler, modelin performansını etkileyebilir, bu nedenle bu tür değişkenlerin sayısını bilmek, modelleme sürecinde nasıl ele alınması gerektiğini anlamamıza yardımcı olur.

# Benzersiz değer sayısı 2'den büyük ve 6'dan küçük veya eşit olan sütunları seçer ve bu sütunların adlarını bir liste olarak saklar.
# Bu, çok kategorili (multiclass) kategorik değişkenleri tanımlamak için kullanılır, çünkü bu tür değişkenler birden fazla kategori içerir (örneğin, 'Kırmızı'/'Sarı'/'Yeşil' gibi).
# Bu liste, daha sonra bu sütunlara özel işlemler uygulamak için kullanılabilir, örneğin one-hot encoding veya label encoding gibi.
categorical_variables = list(df_uniques[(df_uniques['Unique Values Count'] > 2) & (df_uniques['Unique Values Count'] <= 6)].index)

categorical_variables # Çok kategorili (multiclass) kategorik değişkenlerin listesini gösterir.

['offer', 'internet_type', 'contract', 'payment', 'satisfaction']

In [28]:
# Çok kategorili değişkenlerin benzersiz değerlerini göstermek, her bir çok kategorili sütunun hangi kategorileri içerdiğini anlamamıza yardımcı olur. 
# Bu, özellikle modelleme aşamasında hangi sütunların one-hot encoding veya label encoding gerektirebileceğini belirlemek için önemlidir.

# Ayrıca, çok kategorili değişkenlerin benzersiz değerlerini bilmek, modelin karmaşıklığını artırabilecek kategorilerin sayısını ve türünü anlamamıza yardımcı olur. 
# Bu, veri ön işleme stratejilerini planlamak için kritik bir adımdır.

[[cat, list(df[cat].unique())]for cat in categorical_variables] # Çok kategorili değişkenlerin benzersiz değerlerini gösterir. Her bir çok kategorili sütun için, sütun adı ve o sütundaki benzersiz kategorilerin listesini içeren alt listelerden oluşan bir liste oluşturur.

[['offer', ['None', 'Offer E', 'Offer D', 'Offer C', 'Offer B', 'Offer A']],
 ['internet_type', ['DSL', 'Fiber Optic', 'Cable', 'None']],
 ['contract', ['Month-to-Month', 'One Year', 'Two Year']],
 ['payment', ['Bank Withdrawal', 'Credit Card', 'Mailed Check']],
 ['satisfaction', [3, 2, 1, 4, 5]]]

In [29]:
# KNN algoritması, sayısal verilerle çalışır, bu nedenle kategorik değişkenlerin sayısal verilere dönüştürülmesi gerekir.
# Kategorik değişkenleri sayısal verilere dönüştürmek için genellikle one-hot encoding veya label encoding gibi teknikler kullanılır.
# Ancak, bazı durumlarda, özellikle kategorik değişkenler ordinal (sıralı) ise, bu tür değişkenleri sayısal verilere dönüştürmek için pd.cut() fonksiyonu kullanılabilir. 
# Bu fonksiyon, sürekli bir değişkeni belirli aralıklara bölerek kategorik bir değişkene dönüştürür.
# Örneğin, 'months' adlı bir sütun varsa ve bu sütun sürekli bir değişken içeriyorsa, pd.cut() fonksiyonu kullanılarak bu sütun belirli aralıklara bölünebilir ve böylece kategorik bir değişkene dönüştürülebilir.
# Bu, KNN algoritması için uygun bir format sağlar ve modelin performansını artırabilir.

df['months'] = pd.cut(df['months'], bins=5) # 'months' sütununu 5 eşit aralığa bölerek kategorik bir değişkene dönüştürür. bins=5 ifadesi, 'months' sütunundaki değerleri 5 farklı kategoriye ayırır.

In [30]:
# 'months' sütunu, müşterilerin hizmette kalma süresini temsil eder ve bu süre, müşteri davranışını etkileyebilir. Bu nedenle, 'months' sütunu ordinal (sıralı) bir değişken olarak kabul edilir, çünkü daha uzun süre hizmette kalan müşteriler genellikle daha sadık ve değerli müşteriler olarak değerlendirilir.
# 'contract' sütunu, müşterilerin hizmet sözleşmesi türünü temsil eder ve bu tür sözleşmeler genellikle belirli bir süre boyunca hizmeti kullanmayı taahhüt eder. Bu nedenle, 'contract' sütunu da ordinal (sıralı) bir değişken olarak kabul edilir, çünkü daha uzun süreli sözleşmeler genellikle daha sadık müşterileri temsil eder.
# 'satisfaction' sütunu, müşterilerin hizmetten ne kadar memnun olduklarını temsil eder ve bu memnuniyet düzeyi genellikle belirli bir sıralamaya sahiptir (örneğin, 'Düşük', 'Orta', 'Yüksek'). Bu nedenle, 'satisfaction' sütunu da ordinal (sıralı) bir değişken olarak kabul edilir, çünkü daha yüksek memnuniyet düzeyleri genellikle daha sadık müşterileri temsil eder.  
# Bu nedenle, 'months', 'contract' ve 'satisfaction' sütunları ordinal (sıralı) değişkenler olarak kabul edilir ve bu sütunların sayısal verilere dönüştürülmesi için pd.cut() veya benzeri teknikler kullanılabilir.
# Bu sütunların sayısal verilere dönüştürülmesi, KNN algoritmasının bu bilgileri daha etkili bir şekilde kullanmasını sağlar ve modelin performansını artırabilir.

ordinal_variables = ['months', 'contract', 'satisfaction'] # 'months', 'contract' ve 'satisfaction' sütunlarını ordinal (sıralı) değişkenler olarak tanımlar. Bu, bu sütunların belirli bir sıralamaya sahip olduğunu ve bu sıralamanın modelleme sürecinde dikkate alınması gerektiğini belirtir.

In [31]:
# Kategorik ve ikili değişkenler tanımlandıktan sonra, geriye kalan sütunlar sayısal değişkenler olarak kabul edilir. Bu nedenle, sayısal değişkenlerin listesini oluşturmak için, tüm sütunlardan ordinal, kategorik ve ikili değişkenlerin çıkarılması gerekir.
# Bu, sayısal değişkenlerin doğru bir şekilde tanımlanmasını sağlar ve modelleme sürecinde bu değişkenlere uygun işlemler uygulanmasına olanak tanır.
# Ayrıca, sayısal değişkenlerin doğru bir şekilde tanımlanması, modelin performansını artırabilir, çünkü KNN algoritması sayısal verilerle daha etkili bir şekilde çalışır.

numeric_variables = list(
    set(df.columns) - set(ordinal_variables) - set(categorical_variables) - set(binary_variables) # Tüm sütunlardan ordinal, kategorik ve ikili değişkenlerin çıkarılmasıyla sayısal değişkenlerin listesini oluşturur.
) # set() fonksiyonu, her bir listeyi küme (set) veri yapısına dönüştürür ve bu kümeler arasındaki farkı alarak sayısal değişkenlerin listesini oluşturur. Bu, sayısal değişkenlerin doğru bir şekilde tanımlanmasını sağlar. 

numeric_variables # Sayısal değişkenlerin listesini gösterir. Bu liste, veri setindeki tüm sütunlardan ordinal, kategorik ve ikili değişkenlerin çıkarılmasıyla oluşturulur.

['gb_mon', 'monthly']